# 05 — Final Evaluation
**Goal:** evaluate the saved LSTM on held-out videos and save metrics/plots.  
Do not tune model settings using the test results.

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

PROJECT_DIR = Path.cwd()
SEQ_DIR = PROJECT_DIR / "data" / "processed" / "sequences"
MODEL_PATH = PROJECT_DIR / "outputs" / "models" / "lstm_action_model.pt"
REPORT_DIR = PROJECT_DIR / "outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_test = np.nan_to_num(np.load(SEQ_DIR / "X_test.npy"), nan=0.0, posinf=0.0, neginf=0.0)
y_test = np.load(SEQ_DIR / "y_test.npy")
label_map = json.loads((SEQ_DIR / "label_map.json").read_text(encoding="utf-8"))
classes = [label_map[str(i)] for i in range(len(label_map))]

In [ ]:
class PoseLSTM(nn.Module):
    def __init__(self, input_size=51, hidden_size=64, num_layers=1, num_classes=5, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.classifier(self.dropout(h_n[-1]))

ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model = PoseLSTM(num_classes=ckpt["num_classes"]).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

with torch.no_grad():
    logits = model(torch.tensor(X_test, dtype=torch.float32, device=DEVICE))
    y_pred = logits.argmax(1).cpu().numpy()

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, labels=list(range(len(classes))),
                            target_names=classes, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=list(range(len(classes))))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
fig, ax = plt.subplots(figsize=(10,8))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout()
plot_path = REPORT_DIR / "confusion_matrix.png"
plt.savefig(plot_path, dpi=160)
plt.show()
print("Saved confusion matrix:", plot_path.resolve())

report = classification_report(y_test, y_pred, labels=list(range(len(classes))),
                               target_names=classes, output_dict=True, zero_division=0)
import pandas as pd
report_path = REPORT_DIR / "classification_report.csv"
pd.DataFrame(report).transpose().to_csv(report_path)
print("Saved report:", report_path.resolve())